# Notebook 30: Latent Diffusion on LSUN Bedrooms

---

## What This Notebook Covers

This is the **capstone** of the diffusion track: it combines the attention-conditioned diffusion U-Net of notebook 28 with the variational autoencoder idea of notebook 29 into **latent diffusion** — the architecture behind Stable Diffusion. Instead of running an expensive diffusion process on 256×256 pixel images, we first compress each image into a tiny 4×32×32 **latent** with a pretrained VAE (a 48× reduction), run the *entire* diffusion process in that latent space, and finally decode the generated latent back to a full-resolution image. We will learn:

1. **Why latent diffusion** — how compressing to a latent makes high-resolution diffusion tractable in compute and memory
2. **Using a pretrained Stable Diffusion VAE** (`diffusers.AutoencoderKL`) to encode images to latents and decode them back
3. **Precomputing the entire latent dataset to a memory-mapped file** — the out-of-core engineering pattern that makes training on 303,125 images practical
4. **Diffusing in latent space** — reusing `noisify`, `EmbUNetModel`, `ddim_step`, and `sample` from `miniai.diffusion` (notebook 28), now on 4-channel latents
5. **`init_ddpm`** — zero-initializing residual output convs so the model starts near-identity, a standard diffusion-training stabilizer
6. **Latent scaling** — why we rescale latents toward unit variance before the noise schedule (and undo it before decoding)
7. **End-to-end generation** — sample a latent from noise, decode it with the VAE, and get a novel bedroom

---

## Why Latent Diffusion?

Diffusion in pixel space is brutally expensive. Every one of the ~50–1000 sampling steps runs the full U-Net over the *entire* image; at 256×256×3 that is 196,608 numbers per image, and at 512×512 it is four times that. The U-Net's cost — especially the `O(S²)` attention over `S = H·W` spatial positions — scales badly with resolution. This is why early pixel-space diffusion models were small and slow.

The **latent diffusion** insight (Rombach et al., the basis of Stable Diffusion) is a clean factorization of the problem:

1. **Perceptual compression is a solved, cheap, one-time job.** A VAE (or VQ-VAE) can compress an image ~48× into a small latent that preserves everything perceptually important and discards imperceptible high-frequency detail. Train it *once*, freeze it.
2. **Do the hard generative modeling in the small space.** Run the diffusion model on the 4×32×32 latent — 48× fewer numbers, so every U-Net evaluation is dramatically cheaper, attention sequences are short, and you can afford a bigger, better denoiser.
3. **Decode at the end.** One VAE decode turns the generated latent into a full-resolution image.

The two stages are exactly the two notebooks that precede this one: **notebook 29 built the VAE** (here we use a pretrained, better one), and **notebook 28 built the conditioned diffusion U-Net** (here we reuse it verbatim from `miniai.diffusion`). This notebook is where they click together.

**Climate / EO bridges (real ones, and squarely in your wheelhouse).**
- **Latent diffusion ↔ reduced-order generative emulation.** Running a generative model in a compressed latent rather than on the full field is exactly the strategy behind emerging latent-space climate/weather emulators and generative downscalers — diffuse in a learned low-dimensional representation of the atmosphere/surface, decode to the full field. This notebook is the archetype.
- **Memory-mapped precomputation ↔ your HPC/out-of-core workflows.** The `np.memmap` pattern here — precompute an expensive transform of a dataset too big for RAM, store it to disk, stream it during training — is precisely how you handle ERA5 / ECOSTRESS stacks on a SLURM cluster. The deep dive makes this pattern explicit and transferable.
- **VAE encode-once ↔ precomputing embeddings/features** for a large EO corpus before a downstream model.

---

## Prerequisites

You should be comfortable with:

- **The `miniai.diffusion` module** (notebook 28) — `noisify`, `EmbUNetModel`, `ddim_step`, `sample`. We import and reuse them unchanged.
- **The VAE idea** (notebook 29) — encoder → latent → decoder, and that a VAE's latent space is a compact, structured representation. Here we use a *pretrained* Stable Diffusion VAE rather than training our own.
- **DDPM/DDIM** (notebooks 15–20) — the forward noising `xₜ = √ᾱₜ·x₀ + √(1−ᾱₜ)·ε` and DDIM sampling.
- **`numpy` memory-mapping and `DataLoader`s** — helpful; the deep dive explains the memmap.
- **HuggingFace `diffusers`** — we pull a pretrained `AutoencoderKL`.

---


## Google Colab Setup

Run the cells below **once** at the start of each Colab session. They mount Google Drive, set the working directory, install required packages, and clone the `miniai` library from the [fast.ai Part 2 course repo](https://github.com/fastai/course22p2).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir('/content/drive/MyDrive/Fast.AI_Colab')
print(os.getcwd())

In [ ]:
# Install required packages
!pip install -q fastcore fastai diffusers datasets torcheval accelerate einops timm

# Clone the fast.ai Part 2 course repo to get the miniai library
if not os.path.exists('course22p2'):
    !git clone https://github.com/fastai/course22p2.git

# Add miniai to the Python path
import sys
sys.path.insert(0, os.path.join(os.getcwd(), 'course22p2'))

# Verify miniai is accessible
try:
    import miniai
    print(f'miniai loaded successfully from: {miniai.__file__}')
except ImportError:
    print('ERROR: miniai not found. Check that course22p2 was cloned correctly.')

---

*The cells above are Colab-specific setup. The content below is the same as the source notebook (`30_lsun_diffusion-latents_explained.ipynb`).*

> **Heads-up before you run this on Colab (resource warnings).** This notebook is heavy:
> - **LSUN download (`bedroom.tgz`)** is several GB and unpacks to ~303k images. On Colab it goes to ephemeral storage and vanishes at session end &mdash; point `path_data` at a Drive folder if you want it to persist, and expect a long download.
> - **The latent memmap (`data.npmm`)** is **~5 GB** on disk. Colab's free disk is limited; store it on Drive (edit `mmpath`) or subsample the dataset. The encode-once pass over 303k images is also slow on a single Colab GPU.
> - **Training (25 epochs on 300k latents)** will not finish in a free Colab session. Consider a small subset for a smoke test, or load a pretrained checkpoint into `models/lsun_diffusion-latents.pkl` and skip straight to sampling.
>
> The `CUDA_VISIBLE_DEVICES='0'` cell already targets index 0 (correct for Colab's single GPU), so it was left unchanged. No local paths needed rewriting, but the data/memmap locations above are worth pointing at Drive for persistence.

---

# Part 1: Setup and Imports

The key imports are `miniai.diffusion` (everything we built in notebook 28) and `diffusers.AutoencoderKL` (the pretrained VAE). Note we do **not** rebuild the diffusion machinery — it comes in wholesale from `miniai`.

---


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES']='0'

**What does the code above do?**

Pins training to GPU 0 before any CUDA context exists. (Notebook 28 used GPU 1; this one uses 0 — set to whatever your machine has.)


In [ ]:
from miniai.imports import *
from miniai.diffusion import *

from glob import glob
from fastprogress import progress_bar
from diffusers import AutoencoderKL, UNet2DConditionModel

**What does the code above do?**

The two imports that define this notebook:

- **`from miniai.diffusion import *`** — pulls in *everything we built and exported in notebook 28*: `noisify`, `abar`, `EmbUNetModel`, `DownBlock`/`UpBlock`/`EmbResBlock`, `ddim_step`, `sample`, etc. The entire diffusion stack is reused, not rewritten.
- **`from diffusers import AutoencoderKL, UNet2DConditionModel`** — HuggingFace's pretrained VAE class (`AutoencoderKL` is the Stable Diffusion VAE) and a conditional U-Net (imported but not central here).

The `glob` import is for enumerating image files; `progress_bar` drives the latent-precompute loop.

> You may see a Triton warning on import (`A matching Triton is not available…`). It is harmless — Triton is an optional kernel-optimization backend; its absence just means some ops run unoptimized.


In [ ]:
import timm

**What does the code above do?**

Imports `timm` (PyTorch Image Models). Imported by habit across these notebooks; not central here.


In [ ]:
torch.set_printoptions(precision=4, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray_r'
mpl.rcParams['figure.dpi'] = 70

set_seed(42)
if fc.defaults.cpus>8: fc.defaults.cpus=8

**What does the code above do?**

The usual reproducibility and display boilerplate: readable printing, fixed seeds, modest DPI, and an 8-worker cap for data loading.


---

# Part 2: The LSUN Bedroom Data

LSUN Bedrooms is a large (~300k) dataset of bedroom photos — a standard unconditional-generation benchmark. We download it, wrap it in a minimal `Dataset` that reads and center-crops each image to 256×256 RGB, and peek at a batch.

---


In [ ]:
path_data = Path('data')
path_data.mkdir(exist_ok=True)
path = path_data/'bedroom'

**What does the code above do?**

Sets up local paths: a `data/` directory and `data/bedroom` where the extracted images will live.


In [ ]:
url = 'https://s3.amazonaws.com/fast-ai-imageclas/bedroom.tgz'
if not path.exists():
    path_zip = fc.urlsave(url, path_data)
    shutil.unpack_archive('data/bedroom.tgz', 'data')

**What does the code above do?**

Downloads the LSUN bedroom archive (a fast.ai-hosted copy) and unpacks it, but only if `data/bedroom` doesn't already exist. This is a large download; it runs once.


In [ ]:
bs = 64

**What does the code above do?**

Batch size 64 for the *encoding* pass (turning images into latents). Training later uses a different batch size (128) on the small latents.


In [ ]:
def to_img(f): return read_image(f, mode=ImageReadMode.RGB)/255

**What does the code above do?**

Reads an image file into a tensor as 3-channel RGB and scales pixels to `[0, 1]` (dividing by 255). `read_image` / `ImageReadMode` are torchvision utilities.


In [ ]:
class ImagesDS:
    def __init__(self, spec):
        self.path = Path(path)
        self.files = glob(str(spec), recursive=True)
    def __len__(self): return len(self.files)
    def __getitem__(self, i): return to_img(self.files[i])[:, :256,:256]

**What does the code above do?**

A minimal image `Dataset`. `glob(spec, recursive=True)` finds all matching image files; `__getitem__` reads one and **crops it to the top-left 256×256** (`[:, :256, :256]` — all channels, first 256 rows and columns). It is a crop, not a resize, so aspect ratio is preserved and every sample is exactly 3×256×256. No labels — this is unconditional generation.


In [ ]:
ds = ImagesDS(path/f'**/*.jpg')

**What does the code above do?**

Instantiates the dataset over *all* `.jpg` files under `data/bedroom` (the `**` recursive glob descends into subdirectories). This is the full ~300k-image set.


In [ ]:
dl = DataLoader(ds, batch_size=bs, num_workers=fc.defaults.cpus)
xb = next(iter(dl))
show_images(xb[:16], imsize=2)

**What does the code above do?**

Builds a `DataLoader` over the images, grabs one batch, and shows the first 16. **What you should see:** a 4×4 grid of real bedroom photographs — beds, windows, lamps — at 256×256. (Image omitted.)


In [ ]:
xb[:16].shape

torch.Size([16, 3, 256, 256])

**What does the code above do?**

Confirms the batch shape: 16 images, 3 channels, 256×256 — a lot of pixels per image, which is the problem latent diffusion solves.


In [ ]:
16*3*256*256

3145728

**What does the code above do?**

Counts the raw numbers in those 16 images: **3,145,728**. Hold onto this figure — after VAE encoding we will compare it to the latent size and see the compression factor directly.


---

# Part 3: The Pretrained VAE and the Latent Space

Rather than train our own VAE (notebook 29), we load Stable Diffusion's pretrained one and freeze it. We encode images to latents, verify the compression factor, and confirm the latents decode back to recognizable images. First, the big-picture deep dive on *why* we are doing this.

---


## Deep Dive: Latent Diffusion — the Big Picture

Everything in this notebook is organized around one idea: **do the diffusion in a small, learned latent space instead of on raw pixels.** It is worth making the argument precise, because it is the single most important architectural idea in modern image generation.

### The cost problem with pixel-space diffusion

A diffusion model generates by running its denoising U-Net many times (once per sampling step). The U-Net's cost is dominated by two things that both scale badly with image resolution:

- **Convolutions** over `H×W` feature maps: cost ∝ `H·W`.
- **Self-attention** over `S = H·W` positions: cost ∝ `S² = (H·W)²`.

Double the resolution and convolution cost 4×'s while attention cost 16×'s. At 256×256, running hundreds of denoising steps at full resolution is painfully slow; at 512×512 it becomes impractical without enormous compute.

### The factorization

Latent diffusion splits generation into **perceptual compression** and **semantic generation**, and observes that these want *different* tools:

| Stage | Job | Tool | Cost |
|-------|-----|------|------|
| Compression | Throw away imperceptible detail, keep structure | A VAE (trained once, frozen) | One-time |
| Generation | Model the distribution of *content* | Diffusion U-Net | The expensive part — but now in the small space |

A good autoencoder can compress an image ~48× (as we are about to measure) with almost no perceptible loss, because natural images are highly redundant — most of the bits are high-frequency texture the eye barely registers. The VAE learns to keep the perceptually important structure in a compact latent and discard the rest. Crucially, this compression is a **fixed, one-time cost**: train (or download) the VAE once, freeze it, and reuse it for every image forever.

### Why this is such a big win

Run the diffusion U-Net on a 4×32×32 latent instead of a 3×256×256 image:

- **48× fewer numbers** per sample ⇒ every U-Net evaluation is far cheaper, and there are hundreds of them per generated image.
- **Attention sequences shrink** from `256·256 = 65,536` positions to `32·32 = 1,024` — and since attention is `O(S²)`, that is a ~4000× reduction in attention cost at the full-resolution stage. Attention becomes affordable *everywhere* in the latent U-Net.
- **You can afford a bigger denoiser.** The compute saved on resolution can be reinvested in a wider, deeper, more attentive U-Net that models content better.

The result — Stable Diffusion — is why high-quality text-to-image generation runs on a single consumer GPU. This notebook builds the unconditional version of exactly that pipeline: **VAE-encode → diffuse in latent space → VAE-decode.**

### How the two prior notebooks slot in

- **Notebook 29 (VAE)** is the compression stage. Here we swap our hand-trained MLP VAE for Stable Diffusion's convolutional `AutoencoderKL`, which is far better — but the *concept* (encode to a distribution's mean, decode back) is identical, and `latent_dist.mean` below is exactly the `mu` head from notebook 29.
- **Notebook 28 (diffusion U-Net)** is the generation stage, reused verbatim from `miniai.diffusion`. The only change is that it now denoises 4-channel latents instead of 1-channel Fashion-MNIST — a different `in_channels`, nothing more.


In [ ]:
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-ema").cuda().requires_grad_(False)

**What does the code above do?**

Loads Stable Diffusion's pretrained VAE (`sd-vae-ft-ema`, an EMA-finetuned checkpoint) onto the GPU and **freezes it** with `requires_grad_(False)` — we never train it, only use it to encode and decode. `AutoencoderKL` is a convolutional VAE that maps `3×H×W` images to `4×(H/8)×(W/8)` latents (an 8× downsample per spatial dimension). This is the compression stage from the deep dive, downloaded rather than trained.


In [ ]:
xe = vae.encode(xb.cuda())

**What does the code above do?**

Encodes the batch of 256×256 images. `vae.encode` returns an object holding a *distribution* over latents (recall notebook 29: a VAE encoder outputs a distribution, not a point) — specifically `xe.latent_dist`, a diagonal Gaussian with a `.mean` and `.std`.


In [ ]:
xs = xe.latent_dist.mean[:16]
xs.shape

torch.Size([16, 4, 32, 32])

**What does the code above do?**

Takes the **mean** of the encoded distribution for the first 16 images — the deterministic latent code (the analog of `mu` from notebook 29; we use the mean rather than a sample for a stable, reproducible latent). The shape is **`(16, 4, 32, 32)`**: 16 images, 4 latent channels, 32×32 spatial. The 256×256 image collapsed to 32×32 (8× per side) with 4 channels instead of 3.


In [ ]:
(16*3*256*256)/(16*4*32*32)

48.0

**What does the code above do?**

The compression factor, computed directly: `3,145,728 / 65,536 = ` **`48.0`**. Each image is represented by 48× fewer numbers in latent space than in pixel space. *This single number is the reason latent diffusion works* — the diffusion U-Net now operates on 1/48th of the data.


In [ ]:
show_images(((xs[:16,:3])/4).sigmoid(), imsize=2)

**What does the code above do?**

Visualizes the latents themselves (not decoded — the raw latent tensors). We can't show 4 channels as RGB, so it takes the first 3 latent channels, divides by 4, and applies `sigmoid` to squash into `[0,1]` for display. **What you should see:** 16 small, abstract, blotchy 32×32 "images" — a coarse, colorful summary of each bedroom's layout, not a literal thumbnail. The `/4` matters: the SD VAE's latents have a standard deviation around 4–5, so dividing by ~4 brings them into a sensible range before `sigmoid` (foreshadowing the scaling factors we'll use for diffusion).


In [ ]:
xd = to_cpu(vae.decode(xs))
show_images(xd['sample'].clamp(0,1), imsize=2)

**What does the code above do?**

The round-trip check: **decode** the latents back to full 256×256 images with `vae.decode` (which returns a dict; `['sample']` is the reconstructed image tensor), and show them clamped to `[0,1]`. **What you should see:** reconstructions that look essentially like the original bedrooms — proof that the 48× compression preserved the perceptually important content. This is what licenses us to diffuse in the latent space: if the VAE can rebuild an image from its latent, then generating a *new* latent and decoding it will produce a new plausible image.


---

# Part 4: Precomputing All Latents to a Memory-Mapped File

Encoding an image through the VAE is not free, and we will need the latents for *every* image, for *every* epoch of diffusion training. Re-encoding on the fly each epoch would waste enormous compute. Instead we encode the entire dataset **once**, write the latents to a **memory-mapped file** on disk, and stream them during training. This is the engineering pattern that makes training on 300k images practical.

---


In [ ]:
mmpath = Path('data/bedroom/data.npmm')

**What does the code above do?**

Names the file that will hold every latent: `data/bedroom/data.npmm`. `.npmm` is just a convention for "numpy memmap."


In [ ]:
len(ds)

303125

**What does the code above do?**

The dataset size: **303,125** images. That is the number of latents we need to precompute and store.


In [ ]:
mmshape = (303125,4,32,32)

**What does the code above do?**

The shape of the full latent array: `(303125, 4, 32, 32)` — one 4×32×32 latent per image. At float32 (4 bytes) this is `303125·4·32·32·4 ≈ 4.97 GB` on disk — too big to comfortably hold in RAM all at once, which is exactly why we memory-map it.


## Deep Dive: Memory-Mapping the Latent Dataset

This cell block is a small masterclass in the **out-of-core** pattern: working with a dataset that is too large for RAM by keeping it on disk and letting the OS page pieces in and out on demand. If you do HPC work with large climate arrays, this is a pattern you already half-know — here it is, made explicit.

### The problem

We need the latents for all 303,125 images, repeatedly, across 25 epochs. Two naive options both fail:

1. **Re-encode every epoch.** Running the VAE encoder over 300k images each epoch dominates training time — the encode is far more expensive than one diffusion step. Wasteful, since the latents never change (the VAE is frozen).
2. **Encode once into a big in-RAM array.** The array is ~5 GB; on a modest machine, holding that *plus* the model, optimizer, activations, and CUDA buffers risks running out of memory.

### The solution: `np.memmap`

A memory-map is an array whose data lives in a **file on disk**, but which you index exactly like a normal numpy array. The OS transparently loads (pages in) the parts you touch and evicts parts you don't. You get array ergonomics with disk-sized capacity and RAM-sized footprint.

**Writing the latents (the encode-once pass):**

```python
if not mmpath.exists():
    a = np.memmap(mmpath, np.float32, mode='w+', shape=mmshape)   # create a 5GB file, mapped
    i = 0
    for b in progress_bar(dl):                                    # stream image batches
        n = len(b)
        a[i:i+n] = to_cpu(vae.encode(b.cuda()).latent_dist.mean).numpy()  # encode -> write slice
        i += n
    a.flush()                                                     # ensure everything hits disk
    del(a)                                                        # close the map
```

Line by line:

- **`np.memmap(mmpath, np.float32, mode='w+', shape=mmshape)`** creates (or overwrites, `w+`) a disk file sized exactly for `(303125, 4, 32, 32)` float32 values and returns an array-like handle `a`. No 5 GB allocation in RAM — the file *is* the storage.
- **The loop** streams image batches through the frozen VAE encoder, takes `latent_dist.mean` (the deterministic latent), moves it to CPU as numpy, and **writes it into the correct slice** `a[i:i+n]`. The running index `i` tracks position in the big array. Writing to a memmap slice writes to the file (buffered by the OS page cache).
- **`a.flush()`** forces any buffered writes out to physical disk (so a crash after this line doesn't lose data).
- **`del(a)`** releases the mapping.

The whole pass runs **once**, guarded by `if not mmpath.exists()` — subsequent notebook runs skip straight to reading.

### Why this is the right pattern

- **Compute:** the expensive VAE encode happens exactly once, not once per epoch. Training then reads cheap precomputed latents.
- **Memory:** the 5 GB never sits fully in RAM; the OS pages in only the batches currently being trained on, so the resident footprint stays small.
- **Speed:** disk reads of contiguous latent slices are fast, and the OS page cache keeps recently-used chunks hot in RAM automatically.

**Your climate/HPC bridge (this is the same move).** Precomputing an expensive transform of a too-big-for-RAM dataset and memmapping it is exactly how you'd stage, say, normalized ERA5 fields or ECOSTRESS-derived features on a cluster: run the heavy preprocessing once on the nodes, write to a memmap (or a chunked Zarr/HDF5, the same idea), then stream during model training with a tiny memory footprint. The diffusion here is incidental — the pattern is general infrastructure.


In [ ]:
if not mmpath.exists():
    a = np.memmap(mmpath, np.float32, mode='w+', shape=mmshape)
    i = 0
    for b in progress_bar(dl):
        n = len(b)
        a[i:i+n] = to_cpu(vae.encode(b.cuda()).latent_dist.mean).numpy()
        i += n
    a.flush()
    del(a)

**What does the code above do?**

Executes the encode-once pass described in the deep dive: if the memmap file doesn't yet exist, stream every image batch through the frozen VAE, write each batch's latent means into the right slice of the on-disk array, flush, and close. **What you should see:** a progress bar over ~4,700 batches (303,125 / 64), running for a while. On subsequent runs the `if not mmpath.exists()` guard skips this entirely.


In [ ]:
lats = np.memmap(mmpath, dtype=np.float32, mode='r', shape=mmshape)

**What does the code above do?**

Reopens the same file in **read-only** mode (`mode='r'`). `lats` now behaves like a `(303125, 4, 32, 32)` numpy array, but its data streams from disk on access. This is the object we build the training data loaders from.


In [ ]:
b = torch.tensor(lats[:16])

**What does the code above do?**

Reads the first 16 precomputed latents from disk into a tensor — a quick check that the memmap round-tripped correctly.


In [ ]:
xd = to_cpu(vae.decode(b.cuda()))
show_images(xd['sample'].clamp(0,1), imsize=2)

**What does the code above do?**

Decodes those 16 stored latents back to images. **What you should see:** the same bedroom reconstructions as before — confirming the latents were written to and read from disk correctly (garbled images here would mean a shape/stride bug in the memmap). (Image omitted.)


---

# Part 5: Noisifying Latents (and the Scaling Factor)

Now we set up diffusion training data: split the latents into train/valid, and apply the *same* `noisify` from notebook 28 — but to latents instead of pixels. The one new wrinkle is a **scaling factor** applied to the latents, which matters enough to explain.

---


In [ ]:
def collate_ddpm(b): return noisify(default_collate(b)*0.2)

**What does the code above do?**

The collate function for diffusion training. It collates a batch of latents, **multiplies them by 0.2**, and runs the notebook-28 `noisify` on the result (which picks a random noise level and returns `((xₜ, t), ε)`).

Why the `0.2`? The SD VAE's raw latents have a standard deviation around ~4–5 (recall the `/4` in the visualization). But the diffusion noise schedule (`noisify`, `abar`) was designed assuming data with roughly **unit variance** — the forward process mixes signal and unit-scale Gaussian noise via `√ᾱ` and `√(1−ᾱ)`. Feeding it latents 5× too large would put signal and noise on mismatched scales and hurt training. Multiplying by `0.2` (≈ 1/5) rescales the latents to ≈unit variance, matching the schedule's assumption. This is the well-known "latent scaling factor" of latent diffusion (Stable Diffusion uses ~0.18215 for the same reason). We will multiply by `5` (= 1/0.2) to undo it before decoding.


In [ ]:
n = len(lats)

In [ ]:
tds = lats[:n//10*9 ]
vds = lats[ n//10*9:]

**What does the code above do?**

Splits the latents 90/10 into train (`tds`) and validation (`vds`) sets. `n//10*9` is 90% of the count. Both are still memmap views — slicing a memmap yields a memmap, so no 5 GB copy is made.


In [ ]:
bs = 128

**What does the code above do?**

Sets the *training* batch size to 128. Because latents are tiny (4×32×32), we can afford a bigger batch than the 64 used for the pixel-space encoding pass.


In [ ]:
dls = DataLoaders(*get_dls(tds, vds, bs=bs, num_workers=fc.defaults.cpus, collate_fn=collate_ddpm))

**What does the code above do?**

Builds train/valid `DataLoaders` over the latent splits, using the `collate_ddpm` above so each batch arrives already scaled and noisified into `((xₜ, t), ε)` — exactly the format `EmbUNetModel` expects. `get_dls` is a `miniai` helper that wraps datasets in loaders.


In [ ]:
import warnings

In [ ]:
warnings.simplefilter('ignore', UserWarning)

**What does the code above do?**

Silences `UserWarning`s (numpy/torch emit some benign ones when wrapping memmaps in tensors) to keep the training log clean.


In [ ]:
(xt,t),eps = b = next(iter(dls.train))

**What does the code above do?**

Grabs one training batch to inspect. `xt` is the noised, scaled latents; `t` the per-sample noise levels; `eps` the target noise the model must predict — the same `((xₜ,t), ε)` structure as pixel-space diffusion, just with 4-channel latent "images."


In [ ]:
show_images(xt[:9,0], imsize=1.5)

**What does the code above do?**

Shows channel 0 of the first 9 noised latents as grayscale. **What you should see:** noisy, abstract 32×32 blobs — latents partway through the forward diffusion process, at various random noise levels. (Image omitted.)


In [ ]:
xte = vae.decode(xt[:9].cuda()*5)['sample']
show_images(xte.clamp(0,1), imsize=1.5)

**What does the code above do?**

Decodes those *noised* latents back to pixel space to see what partially-noised bedrooms look like. Note the **`*5`** — undoing the `*0.2` scaling from the collate before handing the latents to the VAE (the VAE expects unscaled latents). **What you should see:** blurry, noise-corrupted bedroom images — recognizable structure buried under noise, the pixel-space view of a mid-diffusion latent. (Image omitted.)


---

# Part 6: Training the Latent Diffusion Model

We now train the notebook-28 `EmbUNetModel` to denoise latents. Two things are specific to this setup: `in_channels=out_channels=4` (latents have 4 channels), and a small but important weight-initialization trick, `init_ddpm`.

---


In [ ]:
def init_ddpm(model):
    for o in model.downs:
        for p in o.resnets: p.conv2[-1].weight.data.zero_()

    for o in model.ups:
        for p in o.resnets: p.conv2[-1].weight.data.zero_()

**What does the code above do?**

Zero-initializes the **last layer of the second conv** in every residual block (down-path and up-path). The deep dive explains why this seemingly destructive move helps.


## Deep Dive: `init_ddpm` — Zero-Initializing Residual Outputs

`init_ddpm` walks every `EmbResBlock` in the U-Net and sets `p.conv2[-1].weight.data` — the weight of the *final layer inside the block's second conv* — to **zero**. At first glance zeroing weights sounds like sabotage. It is the opposite: it is a well-known diffusion-training stabilizer.

### What zeroing `conv2`'s last layer does

Recall the `EmbResBlock` forward from notebook 28:

```python
x = self.conv1(x)
x = x*(1+scale) + shift        # FiLM time conditioning
x = self.conv2(x)              # <-- its final layer's weight is zeroed
x = x + self.idconv(inp)       # residual add
```

If the last layer of `conv2` has zero weight, then `conv2(x)` outputs **zero** at initialization, so the block computes:

$$x = 0 + \text{idconv}(\text{inp}) = \text{idconv}(\text{inp}).$$

That is, **every residual block starts as an identity** (or a plain 1×1 projection when channels change). The entire U-Net therefore starts as a near-identity function: it passes its input through almost unchanged, adding nothing.

### Why starting as identity helps diffusion

1. **Clean gradient signal from step one.** The model begins by predicting ≈0 change; the loss immediately reflects "how far is my (currently trivial) prediction from the true noise," and gradients flow into the block weights in a controlled way rather than through a randomly-scrambling initial transform. Training starts stable and the blocks learn their contribution *incrementally* from a sensible baseline.
2. **No early activation blow-up.** A deep stack of randomly-initialized residual blocks can compound and explode activations before the network learns to tame them. Identity-initialized blocks add nothing at first, so activation statistics stay well-behaved from the outset — the same reasoning behind the `1+scale` (not `scale`) trick in notebook 28's FiLM, and behind ResNet's "make the block easy to skip."
3. **It matches the residual structure.** The block is *built* as `output = block(x) + x`; zero-initializing `block` makes the additive branch start empty, which is precisely the regime where residual networks train most reliably.

This trick (zero-init the last conv/norm of each residual branch) is standard in production diffusion codebases. It costs one line and materially improves training stability, especially for the larger model this notebook uses.


In [ ]:
lr = 3e-3
epochs = 25
opt_func = partial(optim.AdamW, eps=1e-5)
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
cbs = [DeviceCB(), ProgressCB(plot=True), MetricsCB(), BatchSchedCB(sched), MixedPrecision()]
model = EmbUNetModel(in_channels=4, out_channels=4, nfs=(128, 256, 512, 768), num_layers=2,
                     attn_start=1, attn_chans=16)
init_ddpm(model)
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)

**What does the code above do?**

Assembles the latent-diffusion training run:

| Component | Choice | Note |
|-----------|--------|------|
| Model | `EmbUNetModel(in_channels=4, out_channels=4, ...)` | **4 channels in/out** — the latent depth, not RGB |
| Widths | `nfs=(128, 256, 512, 768)` | A big U-Net — affordable because the spatial size is only 32×32 |
| Attention | `attn_start=1, attn_chans=16` | Attention on from the 2nd stage down (short sequences ⇒ cheap) |
| Layers | `num_layers=2` per stage | |
| Init | `init_ddpm(model)` | Zero-init residual outputs (deep dive above) |
| Optimizer | AdamW, `eps=1e-5` | Weight decay, unlike notebook 28's Adam |
| Schedule | OneCycle, `max_lr=3e-3`, 25 epochs | |
| Loss | `MSELoss` | Predict the noise `ε` |

This is the notebook-28 U-Net, unchanged in architecture, pointed at 4-channel latents. The whole payoff of latent diffusion is that this large, attention-heavy model is *cheap to run* because it operates on 32×32 latents rather than 256×256 pixels.


In [ ]:
learn.fit(epochs)

**What does the code above do?**

Trains the latent diffusion model for 25 epochs. **What you should see:** MSE loss (between predicted and true noise) descending over training. Because we operate on precomputed latents streamed from the memmap, each epoch is far faster than pixel-space training would be. (Plot/metrics omitted — this is a long run.)


In [ ]:
# torch.save(learn.model.state_dict(), 'models/lsun_diffusion-latents.pkl')

**What does the code above do?**

A commented-out checkpoint save. Uncomment to persist the trained weights to `models/lsun_diffusion-latents.pkl` so you can reload them without retraining. Left commented in the source so a re-run doesn't overwrite an existing checkpoint.


---

# Part 7: Sampling — Generate a Latent, Decode a Bedroom

The finale: run DDIM sampling to generate *latents* from pure noise (reusing notebook 28's `sample`/`ddim_step`), undo the scaling, and **decode with the VAE** to get full-resolution bedrooms. This is the full latent-diffusion pipeline running in reverse: noise → latent → image.

---


In [ ]:
sz = (16,4,32,32)

**What does the code above do?**

The shape of a batch to generate: 16 latents, each 4×32×32. We sample in *latent* space, so the generation target is a latent tensor, not an image.


In [ ]:
# set_seed(42)
preds = sample(ddim_step, model, sz, steps=100, eta=1., clamp=False)

**What does the code above do?**

Runs notebook 28's DDIM `sample` loop to generate 16 latents from noise in 100 steps (`eta=1`, stochastic). One important difference from the Fashion-MNIST run: **`clamp=False`**. In pixel space we clamped the `x₀` estimate to `[-1,1]` (valid pixels); here the target is a *latent*, whose values are not bounded to `[-1,1]`, so clamping would corrupt them. `preds[-1]` is the final generated latent batch (still in the `*0.2`-scaled space the model was trained in). **What you should see:** a progress bar over 100 steps. (HTML progress output only.)


In [ ]:
s = preds[-1]*5

**What does the code above do?**

Takes the final generated latents and **multiplies by 5** to undo the `*0.2` training scale, returning them to the VAE's native latent scale. `s` is now a batch of unscaled, model-generated latents ready to decode.


In [ ]:
with torch.no_grad(): pd = to_cpu(vae.decode(s.cuda()))

**What does the code above do?**

Decodes the generated latents through the frozen VAE (`no_grad`, since it's pure inference) to turn each 4×32×32 latent into a 3×256×256 image. `pd['sample']` holds the decoded bedrooms.


In [ ]:
show_images(pd['sample'][:9].clamp(0,1), imsize=5)

**What does the code above do?**

Displays 9 of the generated, decoded images at a large size. **What you should see:** **novel, synthetic bedroom photographs** — beds, windows, walls, lamps in plausible arrangements — none of which exist in the training set. They may be a touch soft or dreamlike, but they are recognizably bedrooms, generated entirely from noise.

This is the whole pipeline delivering: pure Gaussian noise → 100 DDIM denoising steps in latent space (the notebook-28 U-Net) → a generated 4×32×32 latent → one VAE decode (the notebook-29 idea, pretrained) → a full-resolution bedroom. **Latent diffusion, end to end.**


---

# Summary and What's Next

### What we built

| Stage | What | Reused from |
|-------|------|-------------|
| Compression | Pretrained SD VAE encode/decode (`AutoencoderKL`) | Notebook 29's VAE concept |
| Precompute | All 303k latents → `np.memmap` on disk | Out-of-core engineering |
| Diffusion | `EmbUNetModel` denoising 4-channel latents | Notebook 28 (`miniai.diffusion`), verbatim |
| Stabilizer | `init_ddpm` zero-inits residual outputs | Diffusion-training standard |
| Sampling | DDIM in latent space → VAE decode | Notebooks 20 + 28 + 29 combined |

### The ideas to remember

1. **Latent diffusion = compress once, diffuse in the small space, decode once.** The 48× compression is *the* enabling trick — it makes a big, attention-heavy U-Net cheap to run and is why Stable Diffusion fits on one GPU.
2. **Precompute-and-memmap** the expensive, unchanging transform. Encode the dataset once, stream latents from disk during training. General out-of-core infrastructure — the same move you make with large climate arrays.
3. **Scale latents to ≈unit variance** (`*0.2`) to match the noise schedule's assumption, and undo it (`*5`) before decoding. `clamp=False` when sampling because latents aren't bounded to `[-1,1]`.
4. **`init_ddpm` zero-inits residual outputs** so every block starts as identity and the U-Net starts as a near-identity — stable gradients, no activation blow-up. Same family as `1+scale` and ResNet's skip.
5. **The pieces compose.** Notebook 28 (conditioned diffusion U-Net) + notebook 29 (VAE) = this notebook (latent diffusion). Nothing about the diffusion code changed; it just runs in a different space.

### Where this goes

Notebook 31 (`imgnet_latents`) applies the *same* latent-diffusion pipeline to ImageNet, and adds **class conditioning** (the `CondUNetModel` path from notebook 28) so you can generate a chosen category — the last step toward a fully controllable latent generator. Beyond the course, replacing the class embedding with a *text* embedding and adding cross-attention is exactly Stable Diffusion; replacing images with spatial climate fields and the class label with a physical forcing is the recipe for a latent-space generative climate emulator.

### Suggested next steps

1. Re-read the three deep dives (latent-diffusion big picture, memmap precompute, `init_ddpm`) — the memmap pattern especially is directly reusable in your HPC work.
2. Trace the scaling carefully: `*0.2` in the collate, `*5` before every decode, `/4` in the latent visualization. Confirm you see *why* each is where it is (matching the VAE's ~×5 latent std to the schedule's unit-variance assumption).
3. When satisfied, run `concept-extraction` (candidates: latent diffusion factorization + 48× compression, pretrained `AutoencoderKL` encode/decode, `np.memmap` out-of-core precompute, latent scaling factor, `init_ddpm` zero-init, DDIM `clamp=False` for latents).
4. Optionally `/colab` for a GPU-ready version (note: the LSUN download and the 5 GB memmap are heavy — the deep dives flag this) and `/html` to publish.

---
